In [0]:
# create checkpoints volume to store the checkpoint files (offsets,commits,...) there for each streaming notebook
spark.sql("CREATE VOLUME IF NOT EXISTS catalog_smartFactory.bronze.checkpoints")

DataFrame[]

In [0]:
from pyspark.sql.functions import col, expr

tenant_id = "TENANT-ID"
client_id = "CLIENT-ID"
client_secret = "CLIENT-SECRET"
event_hubs_server = "evnhs-smartFactory-dev.servicebus.windows.net"
event_hubs_topic = "smart-factory-events"

bootstrap_servers = f"{event_hubs_server}:9093"

oauth_scope = f"https://{event_hubs_server}/.default"

jaas_config = (
    "org.apache.kafka.common.security.oauthbearer.OAuthBearerLoginModule required "
    f'clientId="{client_id}" '
    f'clientSecret="{client_secret}" '
    f'scope="{oauth_scope}";'
)

kafka_options = {
    "kafka.bootstrap.servers": f"{event_hubs_server}:9093",

    "subscribe": event_hubs_topic,

    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "OAUTHBEARER",

    "kafka.sasl.jaas.config": (
        "kafkashaded.org.apache.kafka.common.security.oauthbearer."
        "OAuthBearerLoginModule required "
        f'clientId="{client_id}" '
        f'clientSecret="{client_secret}" '
        f'scope="https://{event_hubs_server}/.default";'
    ),

    "kafka.sasl.login.callback.handler.class":
        "kafkashaded.org.apache.kafka.common.security.oauthbearer.secured."
        "OAuthBearerLoginCallbackHandler",

    "kafka.sasl.oauthbearer.token.endpoint.url":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",

    "startingOffsets": "latest",

    "failOnDataLoss": "false"
}

In [0]:
raw_stream_df = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)


### Temporarily write the raw events to a Delta test table

In [0]:
test_query = (
    raw_stream_df
        .selectExpr("CAST(value AS STRING) AS message")
        .writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/catalog_smartfactory/bronze/checkpoints/eventhub_debug_v7"
        )
        .trigger(availableNow=True)
        .toTable("catalog_smartfactory.bronze.eventhub_debug")
)

### Then inspect the result normally

In [0]:
display(
    spark.table("catalog_smartfactory.bronze.eventhub_debug")
)

message
"{""event_id"": ""6a231bb2-5af6-46bc-8eab-312d0cdc3856"", ""timestamp"": ""2026-08-09T11:48:52.824269+00:00"", ""machine_id"": ""MCH-1011"", ""temperature"": 103.69561520556262, ""vibration"": 4.133271723260657, ""pressure"": 108.521686860259, ""rpm"": 4276.260803647834, ""power_cosumption"": 33.718471409289435, ""failure_risk_score"": 0.6570760898626467, ""is_anomaly"": true, ""source"": ""smartFactory-simulator""}"
"{""event_id"": ""387473a7-2f58-4d8e-b4e3-17e30c4d525a"", ""timestamp"": ""2026-08-09T11:48:52.825233+00:00"", ""machine_id"": ""MCH-1027"", ""temperature"": 72.46197629378001, ""vibration"": 0.7702474273150259, ""pressure"": 116.98312837334942, ""rpm"": 3573.3000505483897, ""power_cosumption"": 65.42620930955636, ""failure_risk_score"": 0.1700455909231223, ""is_anomaly"": false, ""source"": ""smartFactory-simulator""}"
"{""event_id"": ""f3477590-f4b9-44e7-ab0e-6cf60f2362cb"", ""timestamp"": ""2026-08-09T11:48:52.825926+00:00"", ""machine_id"": ""MCH-1037"", ""temperature"": 79.24678983996337, ""vibration"": 6.676756803295279, ""pressure"": 126.52077782705264, ""rpm"": 3100.5450513099404, ""power_cosumption"": 26.907078987115156, ""failure_risk_score"": 0.41167312987997734, ""is_anomaly"": true, ""source"": ""smartFactory-simulator""}"
"{""event_id"": ""4163a1bb-370d-4bd6-809f-cb2b96dc41de"", ""timestamp"": ""2026-08-09T11:48:52.826454+00:00"", ""machine_id"": ""MCH-1049"", ""temperature"": 95.39054727556612, ""vibration"": 3.903215988420502, ""pressure"": 139.24841670435924, ""rpm"": 2476.5082543451776, ""power_cosumption"": 53.67298782269607, ""failure_risk_score"": 0.4204508818692334, ""is_anomaly"": true, ""source"": ""smartFactory-simulator""}"
"{""event_id"": ""b0efc6ca-1c74-42e7-a63d-f18de18b3545"", ""timestamp"": ""2026-08-09T11:48:52.826966+00:00"", ""machine_id"": ""MCH-1012"", ""temperature"": 94.3773824987248, ""vibration"": 4.4190153219506545, ""pressure"": 131.26518621388277, ""rpm"": 3567.2911888969093, ""power_cosumption"": 90.08703114635266, ""failure_risk_score"": 0.6461398958207755, ""is_anomaly"": true, ""source"": ""smartFactory-simulator""}"
"{""event_id"": ""04b1b7ad-ef2c-452d-af2f-b2ed5bf5117e"", ""timestamp"": ""2026-08-09T11:48:52.827471+00:00"", ""machine_id"": ""MCH-1023"", ""temperature"": 88.31666583959904, ""vibration"": 5.3106803408022865, ""pressure"": 119.4297182643611, ""rpm"": 2523.672553862705, ""power_cosumption"": 47.51681123898286, ""failure_risk_score"": 0.4493938392835787, ""is_anomaly"": true, ""source"": ""smartFactory-simulator""}"
"{""event_id"": ""6101626b-ed2d-4809-9bf9-a712f60bb0a2"", ""timestamp"": ""2026-08-09T11:48:52.828024+00:00"", ""machine_id"": ""MCH-1014"", ""temperature"": 74.77522080935347, ""vibration"": 1.2807610880779008, ""pressure"": 100.79385551009774, ""rpm"": 3737.657712451433, ""power_cosumption"": 87.2029177923997, ""failure_risk_score"": 0.0075340167440672845, ""is_anomaly"": false, ""source"": ""smartFactory-simulator""}"
"{""event_id"": ""ee4fd6c9-ae93-4bed-9dec-ddb7a1c9a015"", ""timestamp"": ""2026-08-09T11:48:52.828462+00:00"", ""machine_id"": ""MCH-1035"", ""temperature"": 65.47687763143935, ""vibration"": 6.4737839476938515, ""pressure"": 136.15573491973888, ""rpm"": 1508.5224487949663, ""power_cosumption"": 75.34514338446377, ""failure_risk_score"": 0.5553540994221844, ""is_anomaly"": true, ""source"": ""smartFactory-simulator""}"
"{""event_id"": ""fffce82c-5b80-4256-822d-5529af733044"", ""timestamp"": ""2026-08-09T11:48:52.828897+00:00"", ""machine_id"": ""MCH-1014"", ""temperature"": 90.43027337317329, ""vibration"": 4.359510405214978, ""pressure"": 100.51957894394786, ""rpm"": 2680.211374841081, ""power_cosumption"": 39.712139886434265, ""failure_risk_score"": 0.5179677688157549, ""is_anomaly"": true, ""source"": ""smartFactory-simulator""}"
"{""event_id"": ""ee208268-0506-4748-b9bc-455895f35e2a"", ""timestamp"": ""2026-08-09T11:48:52.829443+00:00"", ""machine_id"": ""MCH-1002"", ""temperature"": 60.07706185078193, ""vibration

In [0]:
test_query.stop()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

iot_telemetry_schema = StructType([
    StructField("timestamp",TimestampType()),
    StructField("machine_id",StringType()),
    StructField("temperature",DoubleType()),
    StructField("vibration",DoubleType()),
    StructField("pressure",DoubleType()),
    StructField("rpm",DoubleType()),
    StructField("power_consumption",DoubleType()),
    StructField("failure_risk_score",DoubleType())
])

In [0]:
bronze_stream_df =(
    raw_stream_df
    .selectExpr("CAST(value AS STRING) as json_payload")
    .select(from_json(col("json_payload"), iot_telemetry_schema).alias("data"))
    .select("data.*")
    .withColumn("event_timestamp",to_timestamp("timestamp"))
    .withColumn("ingestion_timestamp",current_timestamp())

)

In [0]:
bronze_query = (
    bronze_stream_df
    .writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "/Volumes/catalog_smartfactory/bronze/checkpoints/bronze_iot_stream_v1"
    )
    .outputMode("Append")
    .trigger(availableNow=True)
    .toTable("catalog_smartfactory.bronze.streaming_iot_telemetry")
)

In [0]:
display(
    spark.table("catalog_smartfactory.bronze.streaming_iot_telemetry")
)

timestamp,machine_id,temperature,vibration,pressure,rpm,power_consumption,failure_risk_score,event_timestamp,ingestion_timestamp
2026-08-09T11:48:52.824Z,MCH-1011,103.69561520556262,4.133271723260657,108.521686860259,4276.260803647834,null,0.6570760898626467,2026-08-09T11:48:52.824Z,2026-08-09T11:52:59.014Z
2026-08-09T11:48:52.825Z,MCH-1027,72.46197629378001,0.7702474273150259,116.98312837334942,3573.3000505483897,null,0.1700455909231223,2026-08-09T11:48:52.825Z,2026-08-09T11:52:59.014Z
2026-08-09T11:48:52.825Z,MCH-1037,79.24678983996337,6.676756803295279,126.52077782705264,3100.5450513099404,null,0.41167312987997734,2026-08-09T11:48:52.825Z,2026-08-09T11:52:59.014Z
2026-08-09T11:48:52.826Z,MCH-1049,95.39054727556612,3.903215988420502,139.24841670435924,2476.5082543451776,null,0.4204508818692334,2026-08-09T11:48:52.826Z,2026-08-09T11:52:59.014Z
2026-08-09T11:48:52.826Z,MCH-1012,94.3773824987248,4.4190153219506545,131.26518621388277,3567.2911888969093,null,0.6461398958207755,2026-08-09T11:48:52.826Z,2026-08-09T11:52:59.014Z
2026-08-09T11:48:52.827Z,MCH-1023,88.31666583959904,5.3106803408022865,119.4297182643611,2523.672553862705,null,0.4493938392835787,2026-08-09T11:48:52.827Z,2026-08-09T11:52:59.014Z
2026-08-09T11:48:52.828Z,MCH-1014,74.77522080935347,1.2807610880779008,100.79385551009774,3737.657712451433,null,0.0075340167440672845,2026-08-09T11:48:52.828Z,2026-08-09T11:52:59.014Z
2026-08-09T11:48:52.828Z,MCH-1035,65.47687763143935,6.4737839476938515,136.15573491973888,1508.5224487949663,null,0.5553540994221844,2026-08-09T11:48:52.828Z,2026-08-09T11:52:59.014Z
2026-08-09T11:48:52.828Z,MCH-1014,90.43027337317329,4.359510405214978,100.51957894394786,2680.211374841081,null,0.5179677688157549,2026-08-09T11:48:52.828Z,2026-08-09T11:52:59.014Z
2026-08-09T11:48:52.829Z,MCH-1002,60.07706185078193,1.4493329061709943,85.78068225739199,4024.5310027644755,null,0.13872097211185686,2026-08-09T11:48:52.829Z,2026-08-09T11:52:59.014Z


In [0]:
bronze_query.stop()